# Training Basics: Step-by-Step Revision

This is a revision notebook for the material covered in [the complete worked reference](../reference/understand_training_basics.ipynb).
We will review your reasoning before moving to momentum. No new optimizer topics are introduced here.

### How to revise

1. Run the setup cell once.
2. Work through **one checkpoint at a time**. Write your answer before running its code.
3. Run the code, compare the result with your prediction, and explain any difference.
4. Open the collapsed answer guide only after attempting the questions.
5. Save your notes and share that checkpoint's answers with your mentor for review.

The code cells are saved unexecuted. Restart the kernel and run from the top if you lose
track of parameter values. Each training experiment resets its own parameters.
Later checkpoints reuse functions and recorded results from earlier checkpoints.

### What we have learned

| Concept | Main idea |
|---|---|
| Parameters | Weight and bias are learned numbers; a weight is one kind of parameter |
| Hyperparameters | We choose settings such as learning rate, batch size, and patience |
| Prediction and loss | Predict with current parameters; mean squared error measures mistakes |
| Gradients | `backward()` calculates how parameter changes affect the current loss |
| Updates | Subtract learning rate times gradient; smaller gradients give smaller plain-SGD updates |
| Gradient clearing | Clear stored gradients while keeping learned parameters |
| Batches, steps, epochs | Combine one batch, update once; an epoch visits the whole dataset |
| Example order | With updates between examples or batches, later examples see changed parameters |
| Training and validation | Training drives updates; validation guides settings and checkpoint selection |
| Data quality | Loss fits supplied targets, including incorrect targets |
| Checkpoints and patience | Save the lowest-validation-loss parameters; patience controls how long to wait |
| Test data | Provide a separate evaluation after choosing settings and checkpoints |
| SGD optimizer | `zero_grad()` clears gradients; `backward()` calculates them; `step()` applies updates |

### Contents

- [1. Parameters, predictions, and loss](#checkpoint-1)
- [2. Gradients and gradient clearing](#checkpoint-2)
- [3. Parameter updates and fresh gradients](#checkpoint-3)
- [4. Learning rate and overshooting](#checkpoint-4)
- [5. Batches, steps, epochs, and order](#checkpoint-5)
- [6. Training loss, validation loss, and incorrect targets](#checkpoint-6)
- [7. Best checkpoints and early stopping](#checkpoint-7)
- [8. A separate test evaluation](#checkpoint-8)
- [9. Manual updates versus plain SGD](#checkpoint-9)

## Chapter links

- Checkpoints 1–4: [Chapter 01 — Training Foundations](../01_training_foundations/README.md).
- Checkpoint 5: [Chapter 02 — Batches and Epochs](../02_batches_and_epochs/README.md).
- Checkpoints 6–8: [Chapter 03 — Validation and Generalization](../03_validation_and_generalization/README.md).
- Checkpoint 9: [Chapter 04 — Optimizers and Momentum](../04_optimizers_and_momentum/README.md), lessons 01–02 (plain SGD).
- [Return to the course index](../README.md).


## Setup: our four examples

PyTorch is already installed for this course. Run this cell to import it and create
the original data. We use `prediction = weight × input + bias`; the correct rule is
`y = 3 × x + 2`. Do not change `x` or `y` during the revision.


In [ ]:
import torch

# Each input is paired with the correct answer at the same position.
x = torch.tensor([0., 1., 2., 3.])
y = torch.tensor([2., 5., 8., 11.])
print("PyTorch:", torch.__version__)
print("Inputs:", x.tolist())
print("Targets:", y.tolist())


## Checkpoint 1: Parameters, predictions, and loss

<a id="checkpoint-1"></a>

**Before running the code:** answer these questions in your own words.


1. In `prediction = 3 × x + 2`, which number is the weight and which is the bias?
2. How many parameters does this model have? Is learning rate a parameter or a hyperparameter?
3. With weight 1 and bias 0, predict all four outputs and calculate their mean squared error.
4. What would loss zero mean for these four examples?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
# The same weight and bias are shared by every example.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
predictions = w * x + b
squared_errors = (predictions - y) ** 2
loss = squared_errors.mean()

print("Predictions:", predictions.detach().tolist())
print("Squared errors:", squared_errors.detach().tolist())
print("Mean loss:", loss.item())


<details>
<summary>Checkpoint 1 answer guide — open after writing your answers</summary>


- Weight is 3; bias is 2. Both are parameters, so there are two parameters.
- Learning rate is a hyperparameter chosen by us.
- Starting predictions: `[0, 1, 2, 3]`. Squared errors: `[4, 16, 36, 64]`.
- Mean squared error: `(4 + 16 + 36 + 64) / 4 = 30`.
- Loss zero here means every prediction equals its target.


</details>


## Checkpoint 2: Gradients and gradient clearing

<a id="checkpoint-2"></a>

**Before running the code:** answer these questions in your own words.


1. Does `requires_grad=True` train a parameter, or enable tracking for gradient calculation?
2. Does `loss.backward()` change the weight and bias?
3. Does setting `.grad = None` clear learned parameters or only their stored gradients? Why clear gradients each step?
4. For input 0, can the weight affect the prediction? For input 2, why is the weight gradient twice the bias gradient?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
# Start fresh; backward() calculates gradients without applying an update.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
loss = ((w * x + b - y) ** 2).mean()
loss.backward()
print("Full-batch gradients:", w.grad.item(), b.grad.item())
print("Parameters after backward:", w.item(), b.item())

# Clearing gradients keeps the parameter values intact.
w.grad = None
b.grad = None
print("Cleared gradients:", w.grad, b.grad)
print("Parameters after clearing:", w.item(), b.item())

# Compare single-example gradients at the SAME starting parameters each time.
for i in (0, 1, 2):
    example_w = torch.tensor(1., requires_grad=True)
    example_b = torch.tensor(0., requires_grad=True)
    example_loss = (example_w * x[i] + example_b - y[i]) ** 2
    example_loss.backward()
    print(
        f"Input {x[i].item():.0f} | weight gradient={example_w.grad.item():.1f} | "
        f"bias gradient={example_b.grad.item():.1f}"
    )


<details>
<summary>Checkpoint 2 answer guide — open after writing your answers</summary>


- `requires_grad=True` enables tracking; it does not train by itself.
- Full-batch gradients are −20 and −10, but parameters remain 1 and 0 after `backward()`.
- Clearing `.grad` preserves parameters. PyTorch adds new gradients to stored gradients,
  so we clear them when we want each step's fresh gradients.
- At input 0, `prediction = bias`; weight has no influence and its gradient is zero.
- At input 1, weight and bias have equal influence and equal gradients.
- At input 2, weight is multiplied by 2, so its gradient is twice the bias gradient.
- These single-example checks reset before each example, unlike sequential training.


</details>


## Checkpoint 3: Parameter updates and fresh gradients

<a id="checkpoint-3"></a>

**Before running the code:** answer these questions in your own words.


1. Starting at weight 1, bias 0, with gradients −20 and −10 and learning rate 0.05, what are the new parameters?
2. Why does a negative gradient cause an increase when we subtract learning rate times gradient?
3. Does `torch.no_grad()` prevent parameters changing, or prevent tracking the update for gradients?
4. Should the next step reuse −20 and −10? What does a fixed learning rate imply about update size?
5. For this squared-error model with plain gradient descent, what happens if every prediction becomes exactly correct?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
loss = ((w * x + b - y) ** 2).mean()
loss.backward()

with torch.no_grad():
    # new parameter = old parameter - learning rate * gradient
    w -= 0.05 * w.grad
    b -= 0.05 * b.grad
    updated_loss = ((w * x + b - y) ** 2).mean().item()
print("After one update:", w.item(), b.item(), "loss:", updated_loss)

# Recalculate gradients from updated parameters using a fresh calculation graph.
w.grad = None
b.grad = None
loss = ((w * x + b - y) ** 2).mean()
loss.backward()
print("Next gradients:", w.grad.item(), b.grad.item())

# Check gradients at an exact fit, using separate parameters.
perfect_w = torch.tensor(3., requires_grad=True)
perfect_b = torch.tensor(2., requires_grad=True)
perfect_loss = ((perfect_w * x + perfect_b - y) ** 2).mean()
perfect_loss.backward()
print("Exact-fit loss and gradients:", perfect_loss.item(), perfect_w.grad.item(), perfect_b.grad.item())


<details>
<summary>Checkpoint 3 answer guide — open after writing your answers</summary>


- First update: weight 2, bias 0.5; loss falls from 30 to 10.25.
- `1 - 0.05 * (-20) = 2`: subtracting a negative adds a positive amount.
- `no_grad()` prevents tracking the update; the parameters still change.
- Recalculate gradients because predictions and errors changed. The next gradients are
  **−11.5 and −6**, yielding next parameters **2.575 and 0.8** at rate 0.05.
- Fixed learning rate does not imply fixed-sized changes: changes depend on current gradients.
- At an exact fit, loss and both gradients are zero. With the plain update rule used here,
  the parameters no longer move. A large learning rate can overshoot before reaching the fit.


</details>


## Checkpoint 4: Learning rate and overshooting

<a id="checkpoint-4"></a>

**Before running the code:** answer these questions in your own words.


1. At identical starting parameters and data, does changing learning rate change the gradients or the update sizes?
2. Which takes a larger step, learning rate 0.01 or 0.05?
3. Can a larger step increase loss even when it follows the negative gradient direction?
4. Does this experiment prove the smallest learning rate is always best?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
def manual_step(rate):
    """Apply one full-batch manual update from the same starting parameters."""
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()
    gradients = (w.grad.item(), b.grad.item())
    with torch.no_grad():
        w -= rate * w.grad
        b -= rate * b.grad
        after = ((w * x + b - y) ** 2).mean().item()
    return w.item(), b.item(), after, gradients

for rate in (0.01, 0.05, 0.50):
    final_w, final_b, final_loss, gradients = manual_step(rate)
    print(
        f"rate={rate:.2f} | gradients={gradients} | weight={final_w:.2f} | "
        f"bias={final_b:.2f} | loss={final_loss:.2f}"
    )


<details>
<summary>Checkpoint 4 answer guide — open after writing your answers</summary>


- Starting gradients stay −20 and −10. Learning rate scales the updates.
- Rate 0.05 takes larger steps than 0.01.
- Loss after one update: 0.01 → 25.21; 0.05 → 10.25; 0.50 → 305.
- The gradient describes local behavior. A large step can overshoot and raise loss.
- Here 0.05 performs better than 0.01; smaller is not automatically better.


</details>


## Checkpoint 5: Batches, steps, epochs, and order

<a id="checkpoint-5"></a>

**Before running the code:** answer these questions in your own words.


1. Does a batch of four examples cause four updates or one update?
2. With four examples, how many updates per epoch do batch sizes 4, 2, and 1 produce?
3. How many epochs at batch size 4 match the four updates made in one epoch at batch size 1? Are examples processed also equal?
4. Why can reversing example order change the batch-size-1 result? Does pair order change a full-batch mean loss at fixed parameters?
5. Does the lowest loss after one epoch prove one batch size is always best?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
def one_epoch(batch_size, reverse=False):
    """Reset a model and visit each example once, updating after each batch."""
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    order = [3, 2, 1, 0] if reverse else [0, 1, 2, 3]
    updates = 0
    for start in range(0, len(order), batch_size):
        indices = order[start:start + batch_size]
        w.grad = None
        b.grad = None
        loss = ((w * x[indices] + b - y[indices]) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= 0.05 * w.grad
            b -= 0.05 * b.grad
        updates += 1
    with torch.no_grad():
        final_loss = ((w * x + b - y) ** 2).mean().item()
    return updates, w.item(), b.item(), final_loss

for size in (4, 2, 1):
    updates, final_w, final_b, final_loss = one_epoch(size)
    print(f"Batch size {size} | updates={updates} | weight={final_w:.4f} | bias={final_b:.4f} | loss={final_loss:.4f}")
updates, final_w, final_b, final_loss = one_epoch(1, reverse=True)
print(f"Batch size 1, reverse order | weight={final_w:.4f} | bias={final_b:.4f} | loss={final_loss:.4f}")


<details>
<summary>Checkpoint 5 answer guide — open after writing your answers</summary>


- One batch contributes to one update, using its combined mean squared error.
- Batch sizes 4, 2, 1 give 1, 2, 4 updates per epoch, respectively.
- Batch size 4 needs 4 epochs for 4 updates. It processes 16 examples in total;
  batch size 1 processes 4 examples in its one epoch. Equal updates and equal data exposure differ.
- Forward losses after one epoch: size 4 → 10.25; size 2 → 1.9777; size 1 → 0.1654.
- Reverse size-1 loss is about 0.4012. Each update changes the parameters for the next example.
- Full-batch pair order does not mathematically change the mean or its gradients at fixed
  parameters, apart from possible rounding differences. Keep inputs paired with their targets.
- Smaller batches made more updates per epoch here; this is not a universal ranking of batch sizes.


</details>


## Checkpoint 6: Training loss, validation loss, and incorrect targets

<a id="checkpoint-6"></a>

**Before running the code:** answer these questions in your own words.


1. Which loss directly produces gradients for parameter updates? What is validation loss used for?
2. Must validation loss decrease every epoch? Does one temporary increase establish overfitting?
3. When a target for input 3 changes from 11 to 14, which answer does the training loss use?
4. Why can fitting an incorrect training target more closely worsen predictions on correct unseen examples?
5. What happened to both losses over 100 epochs with the original targets? How do the changed-target results differ?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
def training_history(targets, epochs=100):
    """Record losses and parameter snapshots for the already-learned training loop."""
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    validation_x = torch.tensor([4., 5.])
    validation_y = torch.tensor([14., 17.])
    history = []
    for epoch in range(1, epochs + 1):
        # Keep parameters between epochs; clear gradients between batches.
        for start in range(0, len(x), 2):
            w.grad = None
            b.grad = None
            loss = ((w * x[start:start + 2] + b - targets[start:start + 2]) ** 2).mean()
            loss.backward()
            with torch.no_grad():
                w -= 0.05 * w.grad
                b -= 0.05 * b.grad
        with torch.no_grad():
            train_loss = ((w * x + b - targets) ** 2).mean().item()
            val_loss = ((w * validation_x + b - validation_y) ** 2).mean().item()
        # Numeric snapshots preserve the parameter values at each epoch.
        history.append({"epoch": epoch, "training_loss": train_loss,
                        "validation_loss": val_loss, "weight": w.item(), "bias": b.item()})
    return history

# Change a COPY so the original data remains correct.
noisy_y = y.clone()
noisy_y[3] = 14.
clean_history = training_history(y)
noisy_history = training_history(noisy_y)

for label, history in [("Original targets", clean_history), ("Changed target", noisy_history)]:
    print("\n" + label)
    for epoch in (1, 2, 5, 100):
        row = history[epoch - 1]
        print(f"Epoch {epoch:3d} | training={row['training_loss']:.6f} | validation={row['validation_loss']:.6f}")


<details>
<summary>Checkpoint 6 answer guide — open after writing your answers</summary>


- Training loss produces update gradients. Validation loss guides settings and checkpoint selection.
- Validation can fluctuate. One temporary increase does not establish overfitting.
- With correct targets, both losses become very small and parameters approach weight 3, bias 2.
- The changed target supplied to loss is 14, regardless of the true rule.
- With that incorrect target, training loss improves overall while validation becomes much worse.
  The parameters move toward fitting the supplied answers, not toward knowledge of which answers are correct.
- Validation examples are evaluated without `backward()` or updates. Their inputs `[4, 5]`
  are also beyond the training range, so parameter errors can affect their predictions more.
- Training loss need not reach zero when the supplied points cannot all fit one straight line.


</details>


## Checkpoint 7: Best checkpoints and early stopping

<a id="checkpoint-7"></a>

**Before running the code:** answer these questions in your own words.


1. A later checkpoint has lower training loss but higher validation loss. Which checkpoint should we prefer for unseen predictions?
2. What does patience count: total epochs, or consecutive epochs without a new best validation loss?
3. Can short patience stop too soon? Did patience 50 improve the changed-target model's best checkpoint?
4. When stopping, should we restore the stopping epoch's parameters or the best saved parameters?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
def replay_early_stopping(history, patience):
    """Review stopping decisions on the recorded trajectory; no new training here."""
    best = None
    waiting = 0
    for row in history:
        if best is None or row["validation_loss"] < best["validation_loss"]:
            best = row.copy()
            waiting = 0
        else:
            waiting += 1
        if waiting >= patience:
            return row["epoch"], best
    return history[-1]["epoch"], best

# Replaying recorded losses gives the same stopping decisions as the early-stopping lesson in Chapter 03.
for label, history in [("Original targets", clean_history), ("Changed target", noisy_history)]:
    for patience in (2, 50):
        stopped, best = replay_early_stopping(history, patience)
        print(f"{label} | patience={patience} | stopped={stopped} | best epoch={best['epoch']} | best validation={best['validation_loss']:.6f}")

# These choices are fixed from validation results before the test evaluation.
_, chosen_clean = replay_early_stopping(clean_history, 50)
_, chosen_noisy = replay_early_stopping(noisy_history, 2)


<details>
<summary>Checkpoint 7 answer guide — open after writing your answers</summary>


- Prefer the checkpoint with lower validation loss for the validation-based selection.
- Patience counts consecutive epochs without a new best validation loss; a new best resets it.
- Original targets: patience 2 stops at 4 and keeps epoch 2. Patience 50 reaches 100 and keeps 100.
- Changed target: patience 2 stops at 3; patience 50 stops at 51. Both keep epoch 1.
- Short patience missed the original run's later recovery. Long patience spent 48 extra epochs
  on the changed-target run without a better saved checkpoint.
- Restore the **best** saved parameter values, not necessarily those at the stopping epoch.
- Early stopping can limit harm from an incorrect target, but correcting that target addresses the cause.


</details>


## Checkpoint 8: A separate test evaluation

<a id="checkpoint-8"></a>

**Before running the code:** answer these questions in your own words.


1. What are the separate roles of training, validation, and test data?
2. If we repeatedly choose settings to improve test loss, is that still an independent final check?
3. Does a low loss on two validation examples guarantee equally low loss on every unseen input?
4. Should evaluating a chosen checkpoint change its parameters?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
# These examples were not used for gradients or validation-based choices.
test_x = torch.tensor([6., 7.])
test_y = torch.tensor([20., 23.])

for label, checkpoint in [("Original targets", chosen_clean), ("Changed target", chosen_noisy)]:
    # Evaluate the saved parameters, not the last global w and b.
    test_w = torch.tensor(checkpoint["weight"])
    test_b = torch.tensor(checkpoint["bias"])
    with torch.no_grad():
        predictions = test_w * test_x + test_b
        test_loss = ((predictions - test_y) ** 2).mean().item()
    print(f"{label} | selected epoch={checkpoint['epoch']} | predictions={predictions.tolist()} | test loss={test_loss:.8f}")


<details>
<summary>Checkpoint 8 answer guide — open after writing your answers</summary>


- Training updates parameters; validation guides model and configuration selection;
  test data evaluates the chosen model after those decisions.
- Repeated test-based tuning makes the test data part of selection too.
- Validation loss estimates performance on reserved examples; it does not guarantee all unseen inputs.
- Evaluation only predicts and measures loss. Parameters stay fixed.
- Original-target test loss is about 0.00000593. Changed-target test loss is about 0.45391288.
- The changed-target checkpoint's weight error is amplified at larger inputs. These tiny held-out
  sets illustrate the roles of the datasets, not performance across every possible example.


</details>


## Checkpoint 9: Manual updates versus plain SGD

<a id="checkpoint-9"></a>

**Before running the code:** answer these questions in your own words.


1. Does an optimizer replace `loss.backward()` or the manual parameter-update lines?
2. What does `[w, b]` tell the optimizer? Does `step()` automatically clear gradients?
3. Put these operations in order: `step()`, calculate predictions/loss, `zero_grad()`, `backward()`.
4. Why initialize parameters and the optimizer before the loop, while clearing gradients inside it?
5. Does the SGD name force us to use batch size 1?


**My answers / prediction:** _Write here._

**After running: observation and explanation:** _Write here, including why the result makes sense._

The next cell checks the behavior discussed in these questions. Run checkpoints in order.


In [ ]:
# Reuse checkpoint 4's manual reference for a one-step comparison.
manual_w, manual_b, manual_after, _ = manual_step(0.05)

w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.05, momentum=0, weight_decay=0)
optimizer.zero_grad(set_to_none=True)
loss = ((w * x + b - y) ** 2).mean()
loss.backward()
print("Parameters after backward, before step:", w.item(), b.item())
optimizer.step()
with torch.no_grad():
    after = ((w * x + b - y) ** 2).mean().item()
print("Manual result:", manual_w, manual_b, manual_after)
print("SGD result:", w.item(), b.item(), after)

# Now reset ONCE and repeat ten full-batch updates with a fresh optimizer.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.05, momentum=0, weight_decay=0)
for step in range(1, 11):
    optimizer.zero_grad(set_to_none=True)
    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        after = ((w * x + b - y) ** 2).mean().item()
    if step in (1, 2, 10):
        print(f"Step {step:2d} | weight={w.item():.4f} | bias={b.item():.4f} | loss={after:.4f}")


<details>
<summary>Checkpoint 9 answer guide — open after writing your answers</summary>


- An optimizer replaces manual parameter updates; `loss.backward()` still calculates gradients.
- `[w, b]` identifies the tensors to update. `step()` does not clear their stored gradients.
- Order: clear gradients → calculate predictions/loss → `backward()` → `step()`.
- Initialization happens before the loop to preserve learned values between steps. Gradients
  are cleared inside the loop so each step uses fresh gradients.
- The optimizer is created for those parameter tensors; create a fresh optimizer when resetting them.
- One plain SGD update matches the manual result: weight 2, bias 0.5, loss 10.25.
- After ten full-batch updates: weight about 3.2916, bias 1.3505, loss 0.1513.
- SGD does not choose your batch size; the examples included in loss determine the batch.
- This revision uses momentum 0 and weight decay 0. Momentum is the next lesson, after review.


</details>


## Review record

Save your written answers above and review them with your mentor one checkpoint at a time.
Numerical results help check your calculations; your explanations show how you understand the process.

| Checkpoint | Review status | Correction or question to revisit |
|---|---|---|
| 1. Parameters and loss | Not yet reviewed | |
| 2. Gradients and clearing | Not yet reviewed | |
| 3. Updates and fresh gradients | Not yet reviewed | |
| 4. Learning rate | Not yet reviewed | |
| 5. Batches and epochs | Not yet reviewed | |
| 6. Training, validation, and data | Not yet reviewed | |
| 7. Checkpoints and patience | Not yet reviewed | |
| 8. Test evaluation | Not yet reviewed | |
| 9. SGD | Not yet reviewed | |

**Next topic after revision:** momentum. Keep the original lesson notebook as your
worked reference and this notebook as your revision and explanation record.
